# Making chloropleth maps in Altair

Here's a quick example of how to make a chloropleth map in Altair.  In this example, we'll work with a fairly large data set of baby names in France from 1900-2019, broken down by department.

To work with geographical data, we'll use the `geopandas`, which loads `pandas` dataframes, but with support for geographical outlines in the `geojson` format.  You can use these dataframes just as you would a regular `pandas` dataframe, but they will include that extra geographical outline data.

To get started, we'll need to import our libraries.

In [1]:
import altair as alt
import pandas as pd
import geopandas as gpd # Requires geopandas -- e.g.: conda install -c conda-forge geopandas
alt.data_transformers.enable('json') # Let Altair/Vega-Lite work with large data sets

pass

# Reading our names data

Now, let's read in our dataset.  The exported data is in CSV format, but with a `;` separator instead of commas.  The INSEE data collapses rare names or where department-level information has been elided (presumably to protect individuals with uncommon names or who were one of the only ones born with that name in a given year).  We'll strip those out.

In [2]:
names = pd.read_csv("dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
names.drop(names[names.dpt == 'XX'].index, inplace=True)

names.sample(5)

,sexe,preusuel,annais,dpt,nombre
3293627,2,NAOMIE,2013,971,3
476589,1,ERWAN,2019,18,3
968744,1,LAURENT,1958,74,20
3463938,2,ROMY,2011,22,3
3026324,2,MARGAUX,2014,13,46


# Loading map data

Next, let's load some map data of regions in France using `geopandas`.  These map data come from the [INSEE] and [IGN] and were processed into the `geojson` format we'll need to work with by [Grégoire David].  Here's the [github] repository.

In this example, we'll work with the simplified departments tiles for the Hexagon, but that repository contains higher-resolution versions, the DOM-TOM, and more.

[Grégoire David]: https://gregoiredavid.fr
[INSEE]: http://www.insee.fr/fr/methodes/nomenclatures/cog/telechargement.asp
[IGN]: https://geoservices.ign.fr/adminexpress
[github]: https://github.com/gregoiredavid/france-geojson/

In [3]:
depts = gpd.read_file('departements-version-simplifiee.geojson')

depts.sample(5)

,code,nom,geometry
42,42,Loire,"POLYGON ((3.89953 46.27591, 3.9094 46.25773, 3..."
84,84,Vaucluse,"MULTIPOLYGON (((4.89291 44.36482, 4.90663 44.3..."
34,34,Hérault,"POLYGON ((3.35836 43.91383, 3.42445 43.9116, 3..."
29,2B,Haute-Corse,"POLYGON ((9.40227 41.8587, 9.37191 41.86501, 9..."
4,05,Hautes-Alpes,"POLYGON ((6.26057 45.12685, 6.29922 45.10855, ..."


Notice how `depts` is a geopandas dataframe.  We'll use it just as a regular `pandas` dataframe, but it includes the geometry info we need to be able to draw those regions when we pass them into Altair.  We just need to make sure that when we work with our data, we keep them in a geopandas dataframe and not a plain dataframe if we want to draw the departments.

In the next cell, notice how we do a right-merge to bring in department data into names.  We do this as a merge on `depts` because we need a geopandas dataframe.  Remember, `depts` is a geopandas dataframe, while `names` is a regular dataframe.  If we did a left merge on `names`, we'd end up with a regular pandas dataframe. After this merge, both `names` and `depts` will be geopandas dataframes.

**Hint:** Be careful when you do your data joins here.  It's easy to accidentally merge the wrong way to accidentally create a _much bigger_ dataset.

In [4]:
# Keep a reference around to the plain pandas dataframe, without geometry data, just in case
just_names = names

names = depts.merge(names, how='right', left_on='code', right_on='dpt')

names.sample(5)

,code,nom,geometry,sexe,preusuel,annais,dpt,nombre
49176,41,Loir-et-Cher,"POLYGON ((0.84122 48.10306, 0.87589 48.10944, ...",1,ALBERT,1905,41,25
1424337,76,Seine-Maritime,"POLYGON ((1.38155 50.06577, 1.40926 50.05707, ...",1,ROGER,1939,76,133
762741,69,Rhône,"POLYGON ((4.38808 46.21979, 4.39205 46.26302, ...",1,JEAN,2014,69,28
52716,32,Gers,"POLYGON ((0.07605 43.98314, 0.14096 43.99468, ...",1,ALBERT,1943,32,14
112899,26,Drôme,"POLYGON ((4.80049 45.29836, 4.8588 45.30895, 4...",1,ANDY,1987,26,3


# Show a name over all years

Now we'll choose a name to show across all years.  To that, we'll group all of the names in a department together (squashing the years together) and use the sum.

In [5]:
grouped = names.groupby(['dpt', 'preusuel', 'sexe'], as_index=False).sum(numeric_only=True)
grouped = depts.merge(grouped, how='right', left_on='code', right_on='dpt') # Add geometry data back in
grouped

,code,nom,geometry,dpt,preusuel,sexe,nombre
0,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,AARON,1,160
1,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABBY,2,3
2,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDALLAH,1,7
3,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDEL,1,3
4,01,Ain,"POLYGON ((4.78021 46.17668, 4.79458 46.21832, ...",01,ABDELKADER,1,3
...,...,...,...,...,...,...,...
239574,NaN,NaN,None,974,ÉSAÏE,1,3
239575,NaN,NaN,None,974,ÉTHAN,1,53
239576,NaN,NaN,None,974,ÉTIENNE,1,3
239577,NaN,NaN,None,974,ÉVA,2,32


Now let's pick a name and check out how it's distribution over the last 120 years across Metropolitan France.  In this example, I choose the name “Lucien,” which I rather like for some reason.

In [6]:
name = 'LUCIEN'
subset = grouped[grouped.preusuel == name]
alt.Chart(subset).mark_geoshape(stroke='white').encode(
    tooltip=['nom', 'code', 'nombre'],
    color='nombre',
).properties(width=800, height=600)

alt.Chart(...)

In [7]:
names04 = names[names.annais == '1954']
grouped = names04.groupby(['preusuel', 'sexe'], as_index=False).sum(numeric_only=True)

grouped["sexe_label"] = grouped["sexe"].map({1: 'Garçon', 2: 'Fille'})
grouped["label"] = grouped["preusuel"] + " - " + grouped["sexe_label"]

options = [1, 2]
labels = ['Garçon', 'Fille']

input_dropdown = alt.binding_radio(
    # Add the empty selection which shows all when clicked
    options=options + [None],
    labels=labels + ['Mixte'],
    name='Sexe: '
)
selection = alt.selection_point(
    fields=['sexe'],
    bind=input_dropdown
)

top_20_2004 = alt.Chart(grouped).mark_bar().encode(
    x = alt.X('nombre:Q', title='Nombre de naissances'),
    y = alt.Y('label:N', sort='-x', title='Prénom / Sexe'),
    color=alt.Color(
        'sexe:N', 
        legend=alt.Legend(title="Sexe", labelExpr="datum.value == '1' ? 'Garçon' : 'Fille'"), 
        scale=alt.Scale(domain=[1,2], range=['steelblue', '#F2A6A0'])
    ),
    tooltip=['label', 'preusuel', 'nombre', 'sexe']
).add_params(
    selection
).transform_filter(
    selection
).transform_window(
    rank='rank(nombre)',
    sort=[alt.SortField('nombre', order='descending')]
).transform_filter(
    alt.datum.rank <= 20,
)


top_20_2004

alt.Chart(...)

In [8]:
names04 = names[names.annais == '2004']
grouped = names04.groupby(['preusuel', 'sexe'], as_index=False).sum(numeric_only=True)

total = grouped['nombre'].sum()

grouped["pct"] = grouped["nombre"] / total * 100

grouped["id"] = grouped["preusuel"] + "_" + grouped["sexe"].astype(str)

bins = [0, 0.001,0.0028,0.0077,0.0215,0.0599,0.1668,0.4642,1.2915,3.5938,10,100]
labels = [
    '0%-0.001%', '0.001–0.0028%', '0.0028–0.0077%', '0.0077–0.0215%', 
    '0.0215–0.0599%', '0.0599–0.1668%', '0.1668–0.4642%', '0.4642–1.2915%', 
    '1.2915–3.5938%', '3.5938–10%', '>10%']
grouped["range"] = pd.cut(
    grouped["pct"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

grouped = grouped.sort_values(["range", "nombre"], ascending=[True, False]).reset_index(drop=True)

n_per_row = 10

grouped["index"] = grouped.groupby("range").cumcount()
grouped["row"] = grouped["index"] // n_per_row
grouped["col"] = grouped["index"] % n_per_row

grouped["row_size"] = grouped.groupby("range")["row"].transform("max") + 1
grouped["x_final"] = grouped["row"] - (grouped["row_size"] - 1) / 2

grouped["y0"] = grouped["range"].astype(str).factorize()[0]
grouped["dy"] = grouped["col"] + 0.5
grouped["y"] = grouped.y0 * n_per_row + grouped.dy 




lines = pd.DataFrame({
    "y": [i * n_per_row for i in range(1, len(labels))]
})
rules = alt.Chart(lines).mark_rule().encode(
    y=alt.Y('y:Q', scale=alt.Scale(type='linear'))
)

selection = alt.selection_interval(bind='scales')

click = alt.selection_point(
    fields=["id"],
    empty=False
)


chart = alt.Chart(grouped).mark_circle().encode(
    x=alt.X("x_final:Q", title='',axis=alt.Axis(ticks=False, labels=False)),
    y=alt.Y(
        "y:Q", 
        title="Étages (% population)", 
        axis=alt.Axis(ticks=False, labels=False)),
    color=alt.Color("range:N"),
    # color=alt.condition(
    #     click,
    #     alt.Color("range:N"),
    #     alt.value("lightgray")
    # ),
    opacity=alt.condition(
        click,
        alt.value(1),
        alt.value(0.15)
    ),
    size=alt.condition(click, alt.value(100), alt.value(20)),
    tooltip=["preusuel", "nombre", "pct", "range", "sexe"]
).add_params(selection, click)



chart + rules


alt.LayerChart(...)